- notebook for creating 3M public dataset by combining other datasets
- first it contains logistic regression to detect proxy features in existing datasets and it removes the obvious ones
- then it combines the preprocessed datasets into one

In [ ]:
import os
import sys
data_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if data_root not in sys.path:
    sys.path.insert(0, data_root)

import pandas as pd
from pathlib import Path

from data.utils.fold_splitting import get_sld, process_url_dataset, SamplerConfig

In [ ]:
FULL_MAL_DATASETS = [p.parent.name for p in Path("full_malicious_datasets").glob("*/dataset.csv")]
MIXED_DATASETS = [p.parent.name for p in Path("good_datasets").glob("*/dataset.csv")]
FULL_BEN_DATASETS = ["benign_commoncrawl"]

COMBINED_PATH = Path("combined_dataset.csv")

def get_dataset_type(dataset_name: str) -> str:
    if dataset_name in FULL_MAL_DATASETS:
        return "full_mal"
    elif dataset_name in FULL_BEN_DATASETS:
        return "full_ben"
    elif dataset_name in MIXED_DATASETS:
        return "mixed"
    else:
        raise Exception("Unknown dataset error")

- create merged dataset of all data for finetuning dataset

In [ ]:
def build_merged_dataset(output_path: Path = COMBINED_PATH) -> tuple[pd.DataFrame, dict]:
    """
    Build merged dataset and return:
      merged_dataframe, original_sizes_dict
    """
    all_dfs = []
    original_sizes: dict[str, int] = {}

    # Benign-only
    benign_csv = Path("benign_commoncrawl/dataset.csv")
    if benign_csv.exists():
        print(f"Reading benign: {benign_csv}")
        df = pd.read_csv(benign_csv, usecols=["url"])
        df["label"] = "benign"
        df["dataset_name"] = "benign_commoncrawl"
        original_sizes["benign_commoncrawl"] = len(df)
        all_dfs.append(df)

    # Malicious-only
    for subcsv in Path("full_malicious_datasets").glob("*/dataset.csv"):
        if not subcsv.exists():
            continue
        print(f"Reading malicious: {subcsv}")
        name = subcsv.parent.name
        df = pd.read_csv(subcsv, usecols=["url"])
        df["label"] = "malicious"
        df["dataset_name"] = name
        original_sizes[name] = len(df)
        all_dfs.append(df)

    # Mixed
    for subcsv in Path("good_datasets").glob("*/dataset.csv"):
        if not subcsv.exists():
            continue
        print(f"Reading mixed: {subcsv}")
        name = subcsv.parent.name
        cols = pd.read_csv(subcsv, nrows=0).columns.tolist()
        usecols = [c for c in ["url", "label"] if c in cols]
        df = pd.read_csv(subcsv, usecols=usecols)
        df["dataset_name"] = name
        original_sizes[name] = len(df)
        all_dfs.append(df)

    if not all_dfs:
        raise FileNotFoundError("No datasets found.")

    merged = pd.concat(all_dfs, ignore_index=True)
    merged.to_csv(output_path, index=False)

    return merged, original_sizes


all_datasets, original_sizes = build_merged_dataset(COMBINED_PATH)
print(all_datasets.info())
print("Original sizes:", original_sizes)

In [ ]:
all_datasets['sld'] = all_datasets['url'].apply(get_sld)

# Split original datasets

In [ ]:
def get_splitted_datasets(datasets_df: pd.DataFrame, target_benign_balance: float = 0.5, max_domain_share: float = 0.1) -> pd.DataFrame:
    """
    Split the datasets into benign and malicious datasets.
    """
    dataset_names = datasets_df["dataset_name"].unique()
    splitted_original_datasets = pd.DataFrame()
    for dataset_name in dataset_names:
        df = datasets_df[datasets_df["dataset_name"] == dataset_name]
        benign_count = (df["label"] == "benign").sum()
        malicious_count = (df["label"] == "malicious").sum()
        min_count = min(benign_count, malicious_count)
        target_count = min_count * 2
        config = SamplerConfig(target_size=target_count, target_benign_balance=target_benign_balance, max_domain_share=max_domain_share)
        processed = process_url_dataset(df, config)
        splitted_original_datasets = pd.concat([splitted_original_datasets, processed])
    return splitted_original_datasets
    
splitted_original_datasets = get_splitted_datasets(all_datasets[all_datasets["dataset_name"].isin(MIXED_DATASETS)])

# Preprocess other datasets

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, precision_score, average_precision_score
from data.utils.feature_extraction import get_easy_url_features
from scipy.stats import spearmanr
from tqdm import tqdm

pd.set_option("display.max_colwidth", None)

def do_logreg(features, labels):
    """
    Train logistic regression and return trained model
    """
    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # Train logistic regression
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(features_scaled, labels)
    
    return lr, scaler

def do_logreg_crossval(features, labels, df_fold_info, dataset_name):
    """
    Perform cross-validation for a single dataset
    """
    eval_folds_for_crossval = [2, 3, 4]
    results = {
        'f1': [],
        'recall': [],
        'precision': [],
        'pr_auc': []
    }
    
    feature_importances = []
    
    for eval_fold in eval_folds_for_crossval:
        print(f"  Fold {eval_fold}: ", end="")
        
        # Split data
        train_mask = df_fold_info != eval_fold
        eval_mask = df_fold_info == eval_fold
        
        if not eval_mask.any():
            print(f"No data in eval fold {eval_fold}, skipping")
            continue
            
        X_train = features[train_mask]
        y_train = labels[train_mask]
        X_eval = features[eval_mask]
        y_eval = labels[eval_mask]
        
        print(f"train={len(X_train)}, test={len(X_eval)}")
        
        # Skip if no positive or negative examples in test set
        if len(np.unique(y_eval)) < 2:
            print(f"    Skipping fold {eval_fold} due to only one class in test set")
            continue
        
        # Train model
        lr, scaler = do_logreg(X_train, y_train)
        
        # Make predictions
        X_eval_scaled = scaler.transform(X_eval)
        y_pred = lr.predict(X_eval_scaled)
        y_pred_proba = lr.predict_proba(X_eval_scaled)[:, 1]  # Probability for malicious class
        
        # Calculate metrics for malicious class (assuming label 1 is malicious)
        f1 = f1_score(y_eval, y_pred, pos_label=1, zero_division=0)
        recall = recall_score(y_eval, y_pred, pos_label=1, zero_division=0)
        precision = precision_score(y_eval, y_pred, pos_label=1, zero_division=0)
        pr_auc = average_precision_score(y_eval, y_pred_proba)
        
        results['f1'].append(f1)
        results['recall'].append(recall)
        results['precision'].append(precision)
        results['pr_auc'].append(pr_auc)
        
        # Store feature coefficients for importance analysis
        feature_importances.append(np.abs(lr.coef_[0]))
    
    return results, feature_importances

def do_logregs_crossvals(df_all, top_n=8):
    """
    Run cross-validation logistic regression for each dataset
    """
    # Set pandas display options for single column output
    df_all = df_all.copy()
    df_all["label"] = df_all["label"].map({"benign": 0, "malicious": 1})
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)
    
    all_names = df_all["dataset_name"].unique()
    
    # Get feature columns from a sample
    print("=== FEATURE SELECTION ===")
    sample_df = df_all[df_all["dataset_name"] == df_all["dataset_name"].iloc[0]]
    df_with_features = get_easy_url_features(sample_df["url"])
    
    # Get feature columns by comparing with original df
    exclude_cols = sample_df.columns.tolist()
    feature_cols = [col for col in df_with_features.columns if col not in exclude_cols]
    
    # Convert boolean columns to int and filter numerical columns
    numerical_cols = []
    for col in feature_cols:
        if df_with_features[col].dtype == "bool":
            numerical_cols.append(col)
        elif np.issubdtype(df_with_features[col].dtype, np.number):
            numerical_cols.append(col)
        else:
            print(f"Skipping non-numerical column: {col} (dtype: {df_with_features[col].dtype})")

    feature_cols = numerical_cols
    print(f"Selected {len(feature_cols)} numerical features:")
    print(f"Features: {feature_cols}")
    print("=" * 50)
    
    all_results = {}
    correlation_results = []
    
    for name in tqdm(all_names):
        print(f"\n=== {name} ===")
        
        df = df_all[df_all["dataset_name"] == name]
        
        # Print dataset size and class balance
        total_size = len(df)
        class_counts = df["label"].value_counts()
        malicious_count = class_counts.get(1, 0)
        benign_count = class_counts.get(0, 0)
        class_ratio = benign_count / (benign_count + malicious_count) if benign_count > 0 else float("inf")

        print(f"Dataset size: {total_size}")
        print(f"Class balance - Malicious: {malicious_count}, Benign: {benign_count}")
        print(f"Benign ratio: {class_ratio:.3f}")
        
        # Extract features using add_easy_url_features
        df_with_features = get_easy_url_features(df["url"])
        df["label"] = df["label"].replace({"benign": 0, "malicious": 1}).astype(int)
        labels = df["label"].values
        fold_info = df["fold"].values
        
        # Convert boolean features to int
        for col in feature_cols:
            if df_with_features[col].dtype == "bool":
                df_with_features[col] = df_with_features[col].astype(int)
        
        features = df_with_features[feature_cols].values
            
        # Run cross-validation
        results, feature_importances = do_logreg_crossval(features, labels, fold_info, name)
        
        # Calculate mean and std for each metric
        def format_metric(values):
            if not values:
                return "N/A"
            mean_val = np.mean(values)
            std_val = np.std(values)
            return f"{mean_val:.3f} ± {std_val:.3f}"
        
        metrics_summary = {}
        for metric, values in results.items():
            metrics_summary[metric] = format_metric(values)
        
        # Calculate mean feature importance across folds and store top features
        top_n_formatted = ""
        if feature_importances:
            mean_importance = np.mean(feature_importances, axis=0)
            
            # Get top N most important features
            top_indices = np.argsort(mean_importance)[-top_n:][::-1]
            
            # Create formatted string for top n features
            top_n_list = []
            for i in range(min(top_n, len(top_indices))):
                idx = top_indices[i]
                feature_name = feature_cols[idx]
                importance_val = mean_importance[idx]
                top_n_list.append(f"{feature_name}: {importance_val:.3f}")
            top_n_formatted = "\n".join(top_n_list)
        
        # Spearman correlation analysis
        corr_row = {"dataset": name}
        for feature in feature_cols:
            correlation, p_value = spearmanr(df_with_features[feature], labels)
            corr_row[feature] = f"{correlation:.3f}"
        correlation_results.append(corr_row)
        
        # Store results
        result_row = {
            "t_c": total_size,
            "m_c": malicious_count,
            "b_c": benign_count,
            "b_rat": f"{class_ratio:.3f}",
            "prec": metrics_summary["precision"],
            "rec": metrics_summary["recall"],
            "f1": metrics_summary["f1"],
            "pr_auc": metrics_summary["pr_auc"],
            "n_folds": len(results["precision"]),
            f"top_{top_n}_features": top_n_formatted,
            "dataset": name
        }
        
        all_results[name] = result_row
    
    print("\n" + "="*60)
    print("LOGISTIC REGRESSION RESULTS")
    print("="*60)
    
    # Create results DataFrame for display
    results_df = pd.DataFrame(list(all_results.values()))
    results_df = results_df.set_index("dataset")
    
    # Create correlation DataFrame
    correlation_df = pd.DataFrame(correlation_results)
    correlation_df = correlation_df.set_index("dataset")
    
    display(results_df.style.set_properties(subset=[f"top_{top_n}_features"], **{"white-space": "pre-wrap"}))
    
    print("\n" + "="*60)
    print("SPEARMAN CORRELATIONS WITH TARGET")
    print("="*60)
    display(correlation_df)
    
    return results_df, correlation_df

# Run the cross-validation analysis
result, correlation_table = do_logregs_crossvals(splitted_original_datasets, top_n=8)

In [ ]:
# calculate features for datasets that are not in MIXED_DATASETS

feature_stats = []
for dataset_name in FULL_BEN_DATASETS + FULL_MAL_DATASETS:
    df = all_datasets[all_datasets["dataset_name"] == dataset_name]
    features = get_easy_url_features(df["url"])
    
    # calculate mean and std for each feature
    features_mean = features.mean()
    features_std = features.std()
    
    # store mean +- std for each feature. Each dataset is row in a table
    row_data = {'dataset': dataset_name}
    for feature in features.columns:
        row_data[feature] = f"{features_mean[feature]:.2f} ± {features_std[feature]:.2f}"
    
    feature_stats.append(row_data)

# Convert to DataFrame and set index
feature_stats_df = pd.DataFrame(feature_stats)
feature_stats_df = feature_stats_df.set_index('dataset')

display(feature_stats_df)

# Protocol preprocessing

## Delete non http / https / empty protocol

In [ ]:
# calculate how many ftp and mailto urls are in the dataset

protocols_to_delete = ("ftp://", "ftps://", "mailto:", "ws://", "wss://")

mask = all_datasets["url"].str.startswith(protocols_to_delete)
masked_datasets = all_datasets[mask]
print("Total number of urls with protocols to delete: ", masked_datasets.shape[0])

for dataset_name in FULL_BEN_DATASETS + FULL_MAL_DATASETS + MIXED_DATASETS:
    counts = []
    masked_dataset_subsample = masked_datasets[masked_datasets["dataset_name"] == dataset_name]
    for protocol in protocols_to_delete:
        counts.append((protocol, masked_dataset_subsample[masked_dataset_subsample["url"].str.startswith(protocol)].shape[0]))
    print(f"{dataset_name}: {counts}")

without_protocols = all_datasets[~mask]

In [ ]:
def remove_http_https(url: str) -> str:
    if url.startswith("http://"):
        return url[7:]
    elif url.startswith("https://"):
        return url[8:]
    else:
        return url

def remove_www(url: str) -> str:
    if url.startswith("www."):
        return url[4:]
    else:
        return url

without_protocols["url"] = without_protocols["url"].apply(remove_http_https)
without_protocols["url"] = without_protocols["url"].apply(remove_www)

In [ ]:
splitted_original_datasets = get_splitted_datasets(without_protocols[without_protocols["dataset_name"].isin(MIXED_DATASETS)])
do_logregs_crossvals(splitted_original_datasets, top_n=8)

- set preprocessing results to be everything result

In [ ]:
all_datasets = without_protocols

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
import tldextract

def get_tld(url: str) -> str:
    suf = tldextract.extract(url if isinstance(url, str) else "").suffix
    return suf if suf else "__blank__"   # keep blanks explicit

def build_tld_counts_json(
    datasets_df: pd.DataFrame,
    dataset_col: str = "dataset_name",
    url_col: str = "url",
    save_path: str | Path = "tld_counts_by_dataset.json",
) -> dict:
    # Precompute TLD once for speed
    df = datasets_df[[dataset_col, url_col]].copy()
    df["_tld"] = df[url_col].map(get_tld)

    out: dict[str, dict] = {}
    total_counts = Counter()

    for name, g in df.groupby(dataset_col, sort=False):
        counts = Counter(g["_tld"])
        out[str(name)] = {tld: int(cnt) for tld, cnt in counts.items()}
        total_counts.update(counts)

    # Add grand total
    out["total"] = {tld: int(cnt) for tld, cnt in total_counts.items()}

    # Save
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    return out

# --- usage ---
url_label_counts = all_datasets.groupby("url")["label"].nunique()
url_label_counts = url_label_counts[url_label_counts > 1].index
without_duplicates_and_conflicts = all_datasets[~all_datasets["url"].isin(url_label_counts)].copy()
without_duplicates_and_conflicts = without_duplicates_and_conflicts.drop_duplicates(subset=["url"])
without_duplicates_and_conflicts = without_duplicates_and_conflicts[without_duplicates_and_conflicts["dataset_name"].isin(MIXED_DATASETS + FULL_MAL_DATASETS)]
tld_json = build_tld_counts_json(without_duplicates_and_conflicts, dataset_col="dataset_name", url_col="url",
                                 save_path="tld_counts_by_dataset.json")
print(list(tld_json.keys())[:5])  # e.g. ['grambeddings', 'threatfox_malurls', ..., 'total']

# Construct dataset

In [ ]:
config = SamplerConfig()
processed = process_url_dataset(all_datasets, config)
processed = processed[["url", "dataset_name", "label", "fold"]]
processed = processed.sample(frac=1.0, random_state=42).reset_index(drop=True)
processed.to_csv("dataset.csv", index=False)

In [ ]:
import pandas as pd

# Suppose you already have original_sizes mapping
# original_sizes = {...}

def display_sources(df):
    total_urls = len(df)

    # Counts and percentages in final dataset
    benign_counts = df[df["label"] == "benign"]["dataset_name"].value_counts()
    benign_perc = df[df["label"] == "benign"]["dataset_name"].value_counts(normalize=True) * 100

    mal_counts = df[df["label"] == "malicious"]["dataset_name"].value_counts()
    mal_perc = df[df["label"] == "malicious"]["dataset_name"].value_counts(normalize=True) * 100

    summary = pd.DataFrame({
        "benign_count": benign_counts,
        "benign_pct": benign_perc,
        "malicious_count": mal_counts,
        "malicious_pct": mal_perc
    }).fillna(0).reset_index().rename(columns={"index": "dataset_name"})

    # Truncate dataset_name
    summary["dataset_name"] = summary["dataset_name"].str.slice(0, 30)

    # Total counts and % of full dataset
    summary["total_count"] = summary["benign_count"] + summary["malicious_count"]
    summary["total_pct"] = (summary["total_count"] / total_urls * 100).map(lambda x: f"{x:.4f}")

    # % of original dataset size used in final dataset
    orig_df = pd.DataFrame(list(original_sizes.items()), columns=["dataset_name", "original_count"])
    summary = summary.merge(orig_df, on="dataset_name", how="left")
    summary["pct_of_original_used"] = (
        summary["total_count"] / summary["original_count"] * 100
    ).map(lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A")

    # Format benign/malicious %
    summary["benign_pct"] = summary["benign_pct"].map(lambda x: f"{x:.4f}")
    summary["malicious_pct"] = summary["malicious_pct"].map(lambda x: f"{x:.4f}")

    display(summary)

display_sources(processed)

In [ ]:
# do logreg on the processed dataset

temp = all_datasets.copy()
temp['dataset_name'] = "final"
processed_balanced = get_splitted_datasets(temp)
do_logregs_crossvals(processed_balanced)